# R19-H203 - Where the exact-k understatement lives: HNSW approximation vs scale

Executor notebook (2026-07-07), **neo4j3 wave-2 graph, READ-ONLY, CPU-only, no LLM / no Bedrock**.

H195(a) found the query-convention shift (exact-k vs generous-top_k-then-truncate) is EXACTLY 0.0 pt at
2798 embeddings (neo4j2) - HNSW is effectively exact there. H188 run B measured a large exact-k penalty.
H203 asks where that penalty lives: is it scale-dependent? We re-run the convention A/B on the larger
wave-2 graph and measure HNSW recall against a brute-force exact-cosine ground truth per k and per
convention.

- **exact-k arm** - query the HNSW index at exactly `top_k = k`, keep top-k
- **generous arm** - query at a generous ceiling `G` then truncate to k
- **ground truth** - brute-force exact cosine over all embeddings (numpy), true top-k per query
- queries derived deterministically from wave-2 article leads (derivation recorded)

## Setup - CPU-only, neo4j3 pinned (explicit driver, config-apnea.yml authoritative)

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""                      # CPU-only, no GPU
import re, json, hashlib, time, datetime, unicodedata
from pathlib import Path
from collections import defaultdict
import numpy as np
from neo4j import GraphDatabase
from dotenv import dotenv_values
from knowledge_graph_foundry import load_settings
from knowledge_graph_foundry.graph.graphrag import vector_query
from rich import print as rprint

ROOT = Path("..")
STAMP = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
LOG = open(ROOT / "logs/h203-h210-scale.log", "a")
def log(m): LOG.write(f"[{datetime.datetime.utcnow().isoformat()}] H203 {m}\n"); LOG.flush(); rprint(m)

settings = load_settings(ROOT / "config-apnea.yml")           # DEF-4: config-over-env precedence, this block authoritative
NEO4J3 = settings.neo4j.uri
_env = dotenv_values(ROOT / ".env")
PW = settings.neo4j.password or _env.get("NEO4J_PASSWORD")     # config first, .env fallback
AUTH = (settings.neo4j.user, PW)
VEC = settings.graphrag.vector_index_name
GENEROUS = 128                                                 # generous top_k ceiling (> max eval k = 64)
KS = [8, 16, 32, 64]
driver3 = GraphDatabase.driver(NEO4J3, auth=AUTH)             # every vector_query uses THIS driver, READ-ONLY
log(f"config uri={NEO4J3} vec={VEC} generous_ceiling={GENEROUS} eval_ks={KS} (CPU-only, neo4j3 pinned, READ-ONLY)")

2026-07-07 22:53:33.749 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


/tmp/ipykernel_2806942/3008285337.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  STAMP = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
/tmp/ipykernel_2806942/3008285337.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  def log(m): LOG.write(f"[{datetime.datetime.utcnow().isoformat()}] H203 {m}\n"); LOG.flush(); rprint(m)


config uri=bolt://user-konrad.jelen-kgf-neo4j3:7687 vec=kgf_entity_embeddings generous_ceiling=128 eval_ks=[8, 16, 
32, 64] (CPU-only, neo4j3 pinned, READ-ONLY)

## Graph pull + fingerprint (H197 pattern: counts + embedding-count digest)

In [2]:
with driver3.session() as s:
    ents = s.run("MATCH (e:Entity) RETURN e.id AS id, e.name AS name, labels(e) AS types").data()
    n_edges = s.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
    n_docs = s.run("MATCH (dd:KGFDocument) RETURN count(dd) AS c").single()["c"]
    emb_rows = s.run("MATCH (e:Entity) WHERE e.embedding IS NOT NULL "
                     "RETURN e.id AS id, e.embedding AS emb").data()
names = {r["id"]: r["name"] for r in ents}
types = {r["id"]: r["types"] for r in ents}
emb_ids = [r["id"] for r in emb_rows]
EMB = np.asarray([r["emb"] for r in emb_rows], dtype=np.float64)     # (N, 1024)
EMB_N = EMB / (np.linalg.norm(EMB, axis=1, keepdims=True) + 1e-12)   # unit-normalized for cosine
idx_of = {i: p for p, i in enumerate(emb_ids)}

def fingerprint():
    dig = ";".join(f"{i}:" + ",".join(f"{x:.4f}" for x in EMB[idx_of[i], :8]) for i in sorted(emb_ids))
    return dict(node_count=len(ents), edge_count=int(n_edges), document_count=int(n_docs),
                embedding_count=len(emb_ids), embedding_dim=EMB.shape[1],
                embedding_digest=hashlib.sha256(dig.encode()).hexdigest()[:16])
t0 = time.time(); FP = fingerprint()
log(f"fingerprint ({(time.time()-t0)*1000:.0f} ms): {json.dumps(FP)}")
log(f"graph: {len(ents)} entities / {n_edges} edges / {n_docs} docs / {len(emb_ids)} embedded (dim {EMB.shape[1]})")

/tmp/ipykernel_2806942/3008285337.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  def log(m): LOG.write(f"[{datetime.datetime.utcnow().isoformat()}] H203 {m}\n"); LOG.flush(); rprint(m)


fingerprint (19 ms): {"node_count": 4649, "edge_count": 26703, "document_count": 952, "embedding_count": 4649, 
"embedding_dim": 1024, "embedding_digest": "8c4267c771d383bd"}

graph: 4649 entities / 26703 edges / 952 docs / 4649 embedded (dim 1024)

## Query derivation from wave-2 article leads (deterministic, no Bedrock)

The vector index stores entity embeddings; we need probe vectors WITHOUT calling the embedder (no Bedrock).
Derivation, recorded per query: sort wave-2 article files by name (deterministic); read each article's lead
(first 300 chars); find the graph entity whose name (>= 4 chars, not a generic stopword) appears in that lead,
preferring the longest such name; use that entity's STORED embedding as the query vector. Collect >= 50
distinct probe entities. This anchors every probe to a real article title/lead while staying embedder-free.

In [3]:
def _n(s): return re.sub(r"\s+", " ", (s or "").casefold())
GEN = set("cpap auto pro plus apap bipap device machine system therapy sleep apnea the and with for "
          "air pressure water hose humidifier mask patient nasal use used using level".split())
# candidate entity names, longest first (greedy longest-match anchoring)
cand = sorted({nid: names[nid] for nid in names if names[nid] and len(_n(names[nid])) >= 4}.items(),
              key=lambda kv: -len(kv[1]))
cand = [(nid, _n(nm)) for nid, nm in cand if idx_of.get(nid) is not None and _n(nm) not in GEN]

wave2_dir = ROOT / "data/interim/apnea-waves/wave_02"
files = sorted(p for p in wave2_dir.iterdir() if p.suffix == ".txt")
queries = []            # (article_file, lead_snippet, entity_id, entity_name)
used = set()
for fp in files:
    lead = _n(fp.read_text(errors="ignore")[:300])
    for nid, gnm in cand:
        if nid in used:
            continue
        if gnm in lead:
            queries.append(dict(article=fp.name, lead=lead[:80], entity_id=nid, entity_name=names[nid]))
            used.add(nid)
            break
    if len(queries) >= 60:
        break
assert len(queries) >= 50, f"only {len(queries)} probe queries derived"
QVEC = np.asarray([EMB_N[idx_of[q["entity_id"]]] for q in queries], dtype=np.float64)
log(f"derived {len(queries)} probe queries from wave-2 article leads (>= 50 required)")
for q in queries[:5]:
    rprint(f"   {q['article']:16} lead='{q['lead'][:40]}...' -> [{q['entity_name']}]")

/tmp/ipykernel_2806942/3008285337.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  def log(m): LOG.write(f"[{datetime.datetime.utcnow().isoformat()}] H203 {m}\n"); LOG.flush(); rprint(m)


derived 60 probe queries from wave-2 article leads (>= 50 required)

c77_d10040.txt   lead='dreamstation heated hose vs hose cover v...' -> [Continuous Positive Airway Pressure]

c77_d10101.txt   lead='dreamstation heated tube/humidifier init...' -> [Continuous Positive Airway Pressure]

c77_d10139.txt   lead='the water thing and the hose **title: cp...' -> [Humidifier water consumption]

c77_d10159.txt   lead='climateline hose **title: airsense 10 cl...' -> [AirSense 10 AutoSet CPAP machine]

c77_d10202.txt   lead='heated hose check **title: heated cpap h...' -> [Heated CPAP Hose Functionality Testing]

## Brute-force exact-cosine ground truth (numpy, all embeddings)

In [4]:
# cosine == dot product on unit-normalized rows; per-query descending order over all N embeddings
SIM = QVEC @ EMB_N.T                                          # (Q, N)
GT_ORDER = np.argsort(-SIM, axis=1)                           # exact ranking, ties by index
GT = {k: [set(emb_ids[j] for j in GT_ORDER[qi, :k]) for qi in range(len(queries))] for k in KS}
log(f"brute-force exact cosine computed: SIM {SIM.shape}, ground-truth top-{max(KS)} per query")

/tmp/ipykernel_2806942/3008285337.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  def log(m): LOG.write(f"[{datetime.datetime.utcnow().isoformat()}] H203 {m}\n"); LOG.flush(); rprint(m)


brute-force exact cosine computed: SIM (60, 4649), ground-truth top-64 per query

## HNSW arms - exact-k vs generous-then-truncate, recall vs exact per k

In [5]:
# generous arm: one HNSW query per probe at top_k = GENEROUS, then truncate to each k
# exact-k arm: one HNSW query per probe at top_k = k
gen_ids = []
for q in queries:
    res = vector_query(driver3, list(EMB_N[idx_of[q["entity_id"]]]), VEC, top_k=GENEROUS)
    gen_ids.append([r["id"] for r in res])
exactk_ids = {k: [] for k in KS}
for k in KS:
    for q in queries:
        res = vector_query(driver3, list(EMB_N[idx_of[q["entity_id"]]]), VEC, top_k=k)
        exactk_ids[k].append([r["id"] for r in res])
log("HNSW arms queried (exact-k at each k; generous once at ceiling)")

/tmp/ipykernel_2806942/3008285337.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  def log(m): LOG.write(f"[{datetime.datetime.utcnow().isoformat()}] H203 {m}\n"); LOG.flush(); rprint(m)


HNSW arms queried (exact-k at each k; generous once at ceiling)

In [6]:
def recall(returned_sets, k):
    return float(np.mean([len(set(returned_sets[qi][:k]) & GT[k][qi]) / k for qi in range(len(queries))]))

matrix = {}
for k in KS:
    r_exact = recall(exactk_ids[k], k)
    r_gen = recall(gen_ids, k)
    matrix[k] = dict(recall_exact_k=round(r_exact, 4), recall_generous_trunc=round(r_gen, 4),
                     gap_pt=round(100 * (r_gen - r_exact), 3))
    log(f"k={k:2d}  exact-k recall={r_exact:.4f}  generous-trunc recall={r_gen:.4f}  "
        f"gap={100*(r_gen-r_exact):+.2f} pt")

gap16 = matrix[16]["gap_pt"]
approx_loss = any(matrix[k]["recall_generous_trunc"] < 0.9999 for k in KS)   # any HNSW under-recall vs exact
truncation_costless = all(matrix[k]["gap_pt"] >= -1e-6 for k in KS)          # generous never below exact-k
bar_gap_ge3 = gap16 >= 3.0
log(f"\napprox loss present (HNSW < exact anywhere) = {approx_loss}")
log(f"truncation costless (generous >= exact-k at every k) = {truncation_costless}")
log(f"k=16 gap = {gap16:+.2f} pt   (H203 bar: >= 3.0 pt at scale)  bar_met={bar_gap_ge3}")

/tmp/ipykernel_2806942/3008285337.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  def log(m): LOG.write(f"[{datetime.datetime.utcnow().isoformat()}] H203 {m}\n"); LOG.flush(); rprint(m)


k= 8  exact-k recall=0.9771  generous-trunc recall=0.9875  gap=+1.04 pt

k=16  exact-k recall=0.9875  generous-trunc recall=0.9958  gap=+0.83 pt

k=32  exact-k recall=0.9917  generous-trunc recall=0.9948  gap=+0.31 pt

k=64  exact-k recall=0.9951  generous-trunc recall=0.9956  gap=+0.05 pt

approx loss present (HNSW < exact anywhere) = True

truncation costless (generous >= exact-k at every k) = True

k=16 gap = +0.83 pt   (H203 bar: >= 3.0 pt at scale)  bar_met=False

## Verdict + report

In [7]:
# scale precondition per the registration: penalty predicted only when index is genuinely approximate
# (>= 10x neo4j2's 2798 embeddings, ~50k+); below ~5k the two conventions are predicted identical.
scale_precondition_met = FP["embedding_count"] >= 5000
if not scale_precondition_met:
    verdict = ("INCONCLUSIVE-FOR-SCALE-PREDICTION: the wave-2 graph carries "
               f"{FP['embedding_count']} embeddings (< the ~5k threshold and far below the ~50k / >=10x "
               "scale the hypothesis requires); the >= 3 pt at-scale prediction cannot be tested here. "
               f"Measured k=16 gap = {gap16:+.2f} pt")
    if abs(gap16) < 0.5:
        verdict += " - the null EXTENDS from 2798 to this scale (convention shift ~0), consistent with " \
                   "run B's gap being a graph-state/render artifact, not a scale law"
elif bar_gap_ge3:
    verdict = f"CONFIRMED at scale: k=16 gap = {gap16:+.2f} pt (>= 3 pt); exact-k understates recall, run B vindicated"
else:
    verdict = f"REFUTED at scale: k=16 gap = {gap16:+.2f} pt (< 3 pt); convention shift ~0 even at this scale"
log(f"\nVERDICT: {verdict}")

report = dict(round="R19-H203", utc=STAMP, graph="neo4j3", uri=NEO4J3, fingerprint=FP,
              generous_ceiling=GENEROUS, eval_ks=KS, n_queries=len(queries),
              query_derivation="wave-2 article leads (first 300 chars) -> longest graph-entity-name match -> "
                               "stored entity embedding as probe vector; no embedder call (no Bedrock)",
              queries=queries,
              recall_matrix={str(k): matrix[k] for k in KS},
              k16_gap_pt=gap16, approx_loss_present=bool(approx_loss),
              truncation_costless=bool(truncation_costless), bar_gap_ge3pt=bool(bar_gap_ge3),
              scale_precondition_met=bool(scale_precondition_met), verdict=verdict,
              deviations=["graph is " + str(FP["embedding_count"]) + " embeddings vs setup message's ~8000 / "
                          "hypothesis's ~50k; scale precondition for the >=3pt prediction NOT met",
                          "probe vectors are stored entity embeddings anchored to article leads (no Bedrock), "
                          "not freshly embedded title strings"])
out = ROOT / f"reports/exact-k-scale-h203-{STAMP}.json"
out.write_text(json.dumps(report, indent=2))
log(f"saved {out}")
LOG.close()

/tmp/ipykernel_2806942/3008285337.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  def log(m): LOG.write(f"[{datetime.datetime.utcnow().isoformat()}] H203 {m}\n"); LOG.flush(); rprint(m)


VERDICT: INCONCLUSIVE-FOR-SCALE-PREDICTION: the wave-2 graph carries 4649 embeddings (< the ~5k threshold and far 
below the ~50k / >=10x scale the hypothesis requires); the >= 3 pt at-scale prediction cannot be tested here. 
Measured k=16 gap = +0.83 pt

saved ../reports/exact-k-scale-h203-20260707T205333Z.json